# Training on the NVD/OSV Dataset

## using labeled.csv

---

Data Loading & Preparation

In [1]:
#import cudf
#print(cudf.Series([1, 2, 3]))

import glob
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd


/home/jaket/miniconda3/envs/data-mining/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("dngchnhtrn/nvd-cve-reports-and-affected-libraries")

print("Path to dataset files:", path)

print(os.listdir(path))

# Load labeled.csv dataset
def load_labeled_data() -> pd.DataFrame:
    df = pd.read_csv(os.path.join(path, "labeled.csv"))
    # Parse target column (space-separated library IDs)
    library_ids_list = []
    for x in df['target']:

        if pd.notna(x):
            parsed_ids = []

            for i in str(x).split():
                parsed_ids.append(int(i))
            library_ids_list.append(parsed_ids)

        else:
            library_ids_list.append([])

    df['library_ids'] = library_ids_list

    # Extract primary library (first in list)
    primary_library_list = []

    for x in df['library_ids']:
        if len(x) > 0:
            primary_library_list.append(x[0])
        else:
            primary_library_list.append(-1)

    df['primary_library'] = primary_library_list

    # Count affected libraries
    num_libraries_list = []
    for x in df['library_ids']:
        num_libraries_list.append(len(x))
    df['num_libraries'] = num_libraries_list

    return df

# Load data
df = load_labeled_data()

df['text'] = df['text'].fillna('')
print("Length with duplicates", len(df))
df = df.drop_duplicates(["text"]).reset_index(drop=True)

unique_labels = sorted(df['primary_library'].unique())
label_map = {label: i+1 for i, label in enumerate(unique_labels)}
df['label'] = df['primary_library'].map(label_map)

print(f"Length without duplicates {len(df)} CVE records")
print(f"Unique primary libraries: {df['primary_library'].nunique()}")
print(f"\nSample record:")
print(df.iloc[1])

Path to dataset files: /home/jaket/.cache/kagglehub/datasets/dngchnhtrn/nvd-cve-reports-and-affected-libraries/versions/2
['nvd.csv', 'osv.csv', 'labeled.csv']
Length with duplicates 159810
Length without duplicates 45813 CVE records
Unique primary libraries: 10520

Sample record:
Unnamed: 0                                                         1
text               documenttemplate package zope attacker dtmldoc...
target                                                             0
library_ids                                                      [0]
primary_library                                                    0
num_libraries                                                      1
label                                                              1
Name: 1, dtype: object


In [3]:
from sklearn.model_selection import train_test_split


counts = df['label'].value_counts()

# Split into single-sample and multi-sample labels
single_sample = df[df['label'].isin(counts[counts == 1].index)]
multi_sample  = df[df['label'].isin(counts[counts > 1].index)]

# Stratified split on multi-sample only
train_multi, test = train_test_split(
    multi_sample,
    test_size=0.2,
    stratify=multi_sample['label'],
    random_state=42
)

# Add single-sample labels back into train only
train = pd.concat([train_multi, single_sample])

X_train = train['text']
y_train = train['label']
X_test  = test['text']
y_test  = test['label']

print(f"Train samples: {len(X_train)}")
print(f"Test samples:  {len(X_test)}")
print(f"Train labels:  {y_train.nunique()}")
print(f"Test labels:   {y_test.nunique()}")

Train samples: 37965
Test samples:  7848
Train labels:  10520
Test labels:   2493


In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

pipeline_tfidf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, sublinear_tf=True))
])

X_train_tfidf = pipeline_tfidf.fit_transform(X_train)
X_test_tfidf  = pipeline_tfidf.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(X_train_tfidf, y_train)

distances, indices = knn.kneighbors(X_test_tfidf)
neighbor_labels = np.array(y_train)[indices]  # shape: (n_test_samples, k)

# For each test sample, check if true label is in top-k predictions
y_test_arr = np.array(y_test)

results = []
for k in range(1, 6):  # evaluate at k=1,2,3,4,5
    top_k_preds = neighbor_labels[:, :k]  # take first k neighbors
    
    correct    = [true in preds for true, preds in zip(y_test_arr, top_k_preds)]
    recall     = sum(correct) / len(correct)                      
    precision  = sum(correct) / (len(correct) * k)                
    f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    results.append({'k': k, 'precision': precision, 'recall': recall, 'f1': round(f1, 3)})

print(pd.DataFrame(results))

   k  precision    recall     f1
0  1   0.775357  0.775357  0.775
1  2   0.417686  0.835372  0.557
2  3   0.287122  0.861366  0.431
3  4   0.219610  0.878440  0.351
4  5   0.177956  0.889781  0.297
